In [27]:
import torch
import clip
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path
from PIL import Image
import numpy as np



In [28]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE

'cuda'

### Load CLIP model and Data


In [29]:
model, preprocess = clip.load("ViT-B/32",device = DEVICE)
model.eval()



CLIP(
  (visual): VisionTransformer(
    (conv1): Conv2d(3, 768, kernel_size=(32, 32), stride=(32, 32), bias=False)
    (ln_pre): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (transformer): Transformer(
      (resblocks): Sequential(
        (0): ResidualAttentionBlock(
          (attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
          )
          (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (mlp): Sequential(
            (c_fc): Linear(in_features=768, out_features=3072, bias=True)
            (gelu): QuickGELU()
            (c_proj): Linear(in_features=3072, out_features=768, bias=True)
          )
          (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        )
        (1): ResidualAttentionBlock(
          (attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
          

In [30]:
DATA_DIR = Path("../data")
IMG_DIR = DATA_DIR / "dress_images"

test = pd.read_csv(DATA_DIR / "dress_test.csv")
print(len(test))

1197


## Building custom dataset

In [31]:
from torch.utils.data import Dataset, DataLoader

class DressImageDataset(Dataset):
    def __init__(self,df, img_dir, preprocess):
        self.df = df.reset_index(drop=True)
        self.img_dir = Path(img_dir)
        self.preprocess = preprocess

    def __len__(self):
        return len(self.df)
    
    def __getitem__(self,idx):
        row = self.df.iloc[idx]
        img_path = self.img_dir/f"{row['item_ID']}.jpg"
        image = Image.open(img_path).convert("RGB")
        return self.preprocess(image),row["item_ID"]


In [32]:
dataset = DressImageDataset(test,IMG_DIR/"test",preprocess)
loader = DataLoader(dataset,batch_size = 64,shuffle = False,num_workers=2)

In [40]:
all_embeddings = []
all_ids = []

with torch.no_grad():
    for images, ids in loader:
        images = images.to(DEVICE)
        feats = model.encode_image(images)
        feats = feats / feats.norm(dim = -1,keepdim=True) # L2 normalize
        all_embeddings.append(feats.cpu())
        all_ids.extend(ids)

image_embeddings = torch.cat(all_embeddings).to(DEVICE)


print(f"Index built: {image_embeddings.shape}")


Index built: torch.Size([1197, 512])


### Text Query Function

In [ ]:
def retrieve(query_text, top_k=6):
    with torch.no_grad():
        tokens = clip.tokenize([query_text], truncate=True).to(DEVICE)
        text_feat = model.encode_text(tokens)
        text_feat = text_feat / text_feat.norm(dim=-1, keepdim=True)
        text_feat = text_feat.cpu()

    scores = (image_embeddings @ text_feat.T).squeeze(1)
    top_idx = scores.topk(top_k).indices.tolist()

    return [(all_ids[i], scores[i].item()) for i in top_idx]

## Visualize

In [ ]:
def show_results(query_text, top_k=6):
    results = retrieve(query_text, top_k)

    fig, axes = plt.subplots(1, top_k, figsize=(16, 4))
    for ax, (item_id, score) in zip(axes, results):
        img_path = IMG_DIR / "test" / f"{item_id}.jpg"
        ax.imshow(mpimg.imread(img_path))
        ax.set_title(f"{score:.3f}", fontsize=9)
        ax.axis("off")

    plt.suptitle(f'Query: "{query_text}"', fontsize=12, y=1.02)
    plt.tight_layout()
    plt.show()

In [43]:
show_results("red floral dress with long sleeves")

AttributeError: 'Tensor' object has no attribute 'items'